<center>
  <font size="10">Shinkansen Travel Experience - Great Learning Hackathon</font>
</center>



## **1. Problem Overview**

The hackathon provided two related datasets for both training and testing:

- **Travel data** — passenger and journey attributes such as age, travel class, distance, and delays.
- **Survey data** — passenger feedback on comfort, service, cleanliness, entertainment, online services, and other aspects of the journey.

The target variable is:

- `Overall_Experience = 1` → Satisfied
- `Overall_Experience = 0` → Not satisfied

The Travel and Survey datasets are linked using `ID`.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **2. Installing Libraries and Loading Datasets**

The necessary libraries are loaded and the four datasets provided are loaded.

```text
data/
├── Traveldata_train_(1).csv
├── Surveydata_train_(1).csv
├── Traveldata_test_(1).csv
└── Surveydata_test_(1).csv
```

In [3]:
import pandas as pd
import numpy as np

# Load
travel_train = pd.read_csv('/content/drive/MyDrive/Python/Traveldata_train_(1).csv')
survey_train = pd.read_csv('/content/drive/MyDrive/Python/Surveydata_train_(1).csv')

travel_test = pd.read_csv('/content/drive/MyDrive/Python/Traveldata_test_(1).csv')
survey_test = pd.read_csv('/content/drive/MyDrive/Python/Surveydata_test_(1).csv')

# Merge
train = pd.merge(travel_train, survey_train, on='ID')
test = pd.merge(travel_test, survey_test, on='ID')

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train['Overall_Experience'] = le.fit_transform(train['Overall_Experience'])

## **3. Feature Engineering**


1. **Total_Delay** — departure delay + arrival delay.
2. **Delay_per_km** — total delay relative to travel distance.
3. **Age_group** — binned passenger age.

In [5]:
def feature_engineering(df):
    df = df.copy()

    # Total delay
    df['Total_Delay'] = df.get('Departure_Delay', 0) + df.get('Arrival_Delay', 0)

    # Delay ratio
    if 'Travel_Distance' in df.columns:
        df['Delay_per_km'] = df['Total_Delay'] / (df['Travel_Distance'] + 1)

    # Age bins
    if 'Age' in df.columns:
        df['Age_group'] = pd.cut(df['Age'], bins=[0,18,35,60,100], labels=[0,1,2,3])

    return df

train = feature_engineering(train)
test = feature_engineering(test)

## **4. Prepare Features and Target**

In [6]:
X = train.drop(['Overall_Experience', 'ID'], axis=1)
y = train['Overall_Experience']

X_test = test.drop(['ID'], axis=1)

In [7]:
cat_cols = X.select_dtypes(include=['object', 'category']).columns

## **5. Train/Validation/Test Split**

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## **6. Prepare Data and Build Model For CatBoost and XGBoost**

CatBoost can work directly with categorical variables. Missing categorical values are converted to an explicit `"Unknown_Cat"` category so the model receives valid categorical values.



In [9]:
!pip install catboost
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import pandas as pd

# Fill NaN values in categorical columns for X and X_test
for col in cat_cols:
    if col in X.columns:
        # Convert to object dtype first to allow adding new string categories
        if isinstance(X[col].dtype, pd.CategoricalDtype):
            X[col] = X[col].astype('object')
        X[col] = X[col].fillna('Unknown_Cat').astype(str)
    if col in X_test.columns:
        # Convert to object dtype first to allow adding new string categories
        if isinstance(X_test[col].dtype, pd.CategoricalDtype):
            X_test[col] = X_test[col].astype('object')
        X_test[col] = X_test[col].fillna('Unknown_Cat').astype(str)

# Apply the same cleaning to X_train and X_val
for col in cat_cols:
    if col in X_train.columns:
        if isinstance(X_train[col].dtype, pd.CategoricalDtype):
            X_train[col] = X_train[col].astype('object')
        X_train[col] = X_train[col].fillna('Unknown_Cat').astype(str)
    if col in X_val.columns:
        if isinstance(X_val[col].dtype, pd.CategoricalDtype):
            X_val[col] = X_val[col].astype('object')
        X_val[col] = X_val[col].fillna('Unknown_Cat').astype(str)

# Handle categorical features for XGBoost
# It's better to encode the whole X and X_test first, then split
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# Align columns - crucial if some categories are in train but not test or vice versa
train_cols = X_encoded.columns
test_cols = X_test_encoded.columns

missing_in_test = set(train_cols) - set(test_cols)
for c in missing_in_test:
    X_test_encoded[c] = 0

missing_in_train = set(test_cols) - set(train_cols)
for c in missing_in_train:
    X_encoded[c] = 0

X_test_encoded = X_test_encoded[train_cols] # Ensure same order

# Convert cat_cols to a list for CatBoost
cat_features_list = cat_cols.tolist()

# CatBoost
cat = CatBoostClassifier(
    iterations=600,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_state=42, # Using a fixed seed for this initial training
    verbose=0
)
cat.fit(X_train, y_train, cat_features=cat_features_list)

# XGBoost
xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42, # Using a fixed seed for this initial training
    eval_metric='logloss'
)
xgb.fit(X_encoded.loc[X_train.index], y_train)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

In [10]:

cat_val = cat.predict_proba(X_val)[:,1]
xgb_val = xgb.predict_proba(X_encoded.loc[X_val.index])[:,1]

## **7. Ensemble Weight and Threshold Search**

In [11]:
from joblib import Parallel, delayed

best_score = 0
best_w = 0
best_t = 0

def evaluate_blend(w, t, cat_val, xgb_val, y_val):
    pred = (w * cat_val + (1 - w) * xgb_val) > t
    return accuracy_score(y_val, pred), w, t

weights = np.arange(0.45, 0.56, 0.01)
thresholds = np.arange(0.47, 0.53, 0.01)

results = Parallel(n_jobs=-1)(delayed(evaluate_blend)(w, t, cat_val, xgb_val, y_val) for w in weights for t in thresholds)

for acc, w, t in results:
    if acc > best_score:
        best_score = acc
        best_w = w
        best_t = t

print("Best Weight:", best_w)
print("Best Threshold:", best_t)
print("Best Validation Accuracy:", best_score)

Best Weight: 0.45
Best Threshold: 0.49
Best Validation Accuracy: 0.9582538673447765


In [12]:
BEST_WEIGHT = best_w
BEST_THRESHOLD = best_t

The hackathon solution blended the probability predictions from CatBoost and XGBoost.

For each combination:

```text
blended_probability =
    weight × CatBoost_probability
    + (1 - weight) × XGBoost_probability
```

A classification threshold was then searched to convert the blended probability into the final class.

The original hackathon search produced:

- **Best weight:** 0.45
- **Best threshold:** 0.49
- **Validation accuracy:** 95.82%

## **8. Final Ensemble Training**

In [13]:
cat_preds = []

for seed in [42, 52]:

    cat_model = CatBoostClassifier(
        iterations=600,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_state=seed,
        verbose=0
    )

    cat_model.fit(X, y, cat_features=cat_features_list)

    pred = cat_model.predict_proba(X_test)[:,1]
    cat_preds.append(pred)

cat_test = np.mean(cat_preds, axis=0)

In [14]:
from joblib import Parallel, delayed
from xgboost import XGBClassifier

def train_and_predict_xgb(seed, X_encoded_data, y_data, X_test_encoded_data):
    xgb_model = XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        eval_metric='logloss'
    )

    xgb_model.fit(X_encoded_data, y_data)
    pred = xgb_model.predict_proba(X_test_encoded_data)[:,1]
    return pred

xgb_preds = Parallel(n_jobs=-1)(delayed(train_and_predict_xgb)(seed, X_encoded, y, X_test_encoded) for seed in [42, 52])

xgb_test = np.mean(xgb_preds, axis=0)

## **9. Generate Final Predictions and Submission**

In [15]:
final_pred = (BEST_WEIGHT * cat_test + (1 - BEST_WEIGHT) * xgb_test) > BEST_THRESHOLD

In [16]:
final_pred = le.inverse_transform(final_pred.astype(int))

In [17]:
import pandas as pd

submission = pd.DataFrame({
    'ID': test['ID'],
    'Overall_Experience': final_pred
})

submission.to_csv('submission_final1.csv', index=False)

## **10. Solution Summary**

### Modeling approach

| Component | Approach |
|---|---|
| Data integration | Merge Travel + Survey data using `ID` |
| Feature engineering | Total delay, delay per km, age groups |
| Model 1 | CatBoost |
| Model 2 | XGBoost |
| Ensemble | Weighted probability blend |
| Weight search | Validation-based grid search |
| Threshold search | Validation-based grid search |
| Final ensemble | Average predictions across two random seeds |
| Evaluation used during hackathon | Accuracy |



## **11. Key Takeaways**

- Combining travel attributes with passenger survey feedback provides a strong basis for satisfaction prediction.
- Feature engineering around **journey delays** adds information beyond the raw delay columns.
- CatBoost is useful for the many categorical survey variables.
- XGBoost provides a complementary tree-based model using one-hot encoded features.
- Blending the two models improved the validation result during the hackathon.
- Searching both the **ensemble weight** and **classification threshold** was an important part of the final solution.

## **12. Limitations and Reproducibility**

This repository documents a hackathon solution rather than a production deployment.

- Competition datasets are kept outside the repository unless redistribution is explicitly permitted.
- The original hackathon notebook was incomplete; this version reconstructs the missing preprocessing/model setup while preserving the documented modeling strategy.
- The recorded **95.87%** figure is the original hackathon validation result.
- Exact reruns can vary slightly with Python/library versions and preprocessing implementation.

## **13. Future Improvements**

- Add systematic cross-validation for ensemble weight selection.
- Compare additional boosting algorithms.
- Perform feature-importance and SHAP analysis.
- Calibrate probabilities before threshold selection.
- Add experiment tracking for model versions and validation results.